# Введение в обучение с подкреплением и Q-обучение

**ML8 · Reinforcement Learning** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~21 min

## Введение

![Краткое содержание обучения с подкреплением в машинном обучении в виде скетчноута](ms-assets/ml/ml-reinforcement.94024374d63348db.webp)
> Скетчноут от [Tomomi Imura](https://www.twitter.com/girlie_mac)

Обучение с подкреплением включает три важных понятия: агент, некоторые состояния и набор действий для каждого состояния. Выполняя действие в заданном состоянии, агент получает награду. Снова представьте себе компьютерную игру Super Mario. Вы — Марио, вы на уровне игры, стоите рядом с краем обрыва. Над вами монета. Вы — Марио, на уровне игры, в конкретной позиции ... это ваше состояние. Сделать шаг вправо (действие) приведёт к тому, что вы упадёте с обрыва, и получите низкий числовой балл. Однако нажатие кнопки прыжка позволит вам заработать очко и остаться в живых. Это положительный исход, и за него вам должна быть присуждена положительная числовая награда.

Используя обучение с подкреплением и симулятор (игру), вы можете научиться играть так, чтобы максимизировать награду — оставаться в живых и набирать как можно больше очков.

> [Видео](https://www.youtube.com/watch?v=lDq_en8RNOo)

> Нажмите на изображение выше, чтобы послушать Дмитрия, рассказывающего об обучении с подкреплением



## Требования и настройка

В этом уроке мы будем экспериментировать с кодом на Python. Вы должны иметь возможность запустить код Jupyter Notebook из этого урока либо на своём компьютере, либо где-то в облаке.

Вы можете открыть [блокнот урока](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/notebook.ipynb) и пройти этот урок для практики.

> **Примечание:** Если вы открываете этот код из облака, вам также нужно скачать файл [`rlboard.py`](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/rlboard.py), который используется в коде блокнота. Поместите его в ту же папку, что и блокнот.



## Введение

В этом уроке мы исследуем мир **[Петра и Волка](https://ru.wikipedia.org/wiki/Петя_и_волк)**, вдохновлённый музыкальной сказкой русского композитора, [Сергей Прокофьев](https://ru.wikipedia.org/wiki/Сергей_Прокофьев). Мы будем использовать **обучение с подкреплением**, чтобы позволить Петру исследовать окружение, собирать вкусные яблоки и избегать встречи с волком.

**Обучение с подкреплением** (RL) — это метод обучения, позволяющий нам узнать оптимальное поведение **агента** в некоторой **среде** путём проведения множества экспериментов. Агент в этой среде должен иметь некоторую **цель**, определённую через **функцию награды**.



## Среда

Для простоты давайте представим мир Петра в виде квадратной доски размером `width` x `height`, вот такой:

![Среда Петра](ms-assets/ml/environment.40ba3cb66256c93f.webp)

Каждая ячейка в этой доске может быть:

* **землёй**, по которой Петя и другие существа могут ходить.
* **водой**, по которой, очевидно, ходить нельзя.
* **деревом** или **травой**, местом, где можно отдохнуть.
* **яблоком**, которое Петя будет рад найти, чтобы накормить себя.
* **волком**, который опасен и которого следует избегать.

Существует отдельный модуль Python, [`rlboard.py`](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/rlboard.py), который содержит код для работы с этой средой. Поскольку этот код не важен для понимания наших концепций, мы импортируем модуль и используем его для создания примерной доски (кодовый блок 1):



In [ ]:
import urllib.request, os
if not os.path.exists('rlboard.py'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/microsoft/ML-For-Beginners/main/8-Reinforcement/1-QLearning/rlboard.py', 'rlboard.py')
from rlboard import *

width, height = 8,8
m = Board(width,height)
m.randomize(seed=13)
m.plot()

Этот код должен вывести изображение среды, похожее на приведённое выше.



## Действия и политика

В нашем примере целью Петра будет найти яблоко, избегая волка и других препятствий. Для этого он по сути может ходить, пока не найдёт яблоко.

Следовательно, в любой позиции он может выбрать одно из следующих действий: вверх, вниз, влево и вправо.

Мы определим эти действия как словарь и сопоставим их с парами соответствующих изменений координат. Например, движение вправо (`R`) будет соответствовать паре `(1,0)`. (кодовый блок 2):



In [ ]:
actions = { "U" : (0,-1), "D" : (0,1), "L" : (-1,0), "R" : (1,0) }
action_idx = { a : i for i,a in enumerate(actions.keys()) }

В итоге стратегия и цель этого сценария таковы:

- **Стратегия** нашего агента (Петра) определяется так называемой **политикой**. Политика — это функция, возвращающая действие для любого заданного состояния. В нашем случае состояние задачи представлено доской, включая текущую позицию игрока.

- **Цель** обучения с подкреплением — в конечном итоге научиться хорошей политике, которая позволит эффективно решать задачу. Однако, для базового варианта рассмотрим самую простую политику, называемую **случайным блужданием**.



## Случайное блуждание

Давайте сначала решим нашу задачу, реализовав стратегию случайного блуждания. При случайном блуждании мы будем случайно выбирать следующее действие из разрешённых, пока не достигнем яблока (кодовый блок 3).

1. Реализуйте случайное блуждание с помощью следующего кода:

    ```python
    def random_policy(m):
        return random.choice(list(actions))
    
    def walk(m,policy,start_position=None):
        n = 0 # количество шагов
        # установить начальную позицию
        if start_position:
            m.human = start_position 
        else:
            m.random_start()
        while True:
            if m.at() == Board.Cell.apple:
                return n # успешно!
            if m.at() in [Board.Cell.wolf, Board.Cell.water]:
                return -1 # съеден волком или утонул
            while True:
                a = actions[policy(m)]
                new_pos = m.move_pos(m.human,a)
                if m.is_valid(new_pos) and m.at(new_pos)!=Board.Cell.water:
                    m.move(a) # выполнить фактическое движение
                    break
            n+=1
    
    walk(m,random_policy)
    ```

    Вызов `walk` должен возвращать длину соответствующего пути, которая может варьироваться от запуска к запуску.

1. Запустите эксперимент блуждания несколько раз (например, 100), и выведите статистику (кодовый блок 4):

    ```python
    def print_statistics(policy):
        s,w,n = 0,0,0
        for _ in range(100):
            z = walk(m,policy)
            if z<0:
                w+=1
            else:
                s += z
                n += 1
        print(f"Average path length = {s/n}, eaten by wolf: {w} times")
    
    print_statistics(random_policy)
    ```

    Обратите внимание, что средняя длина пути около 30-40 шагов, что довольно много, учитывая, что среднее расстояние до ближайшего яблока около 5-6 шагов.

    Также вы можете увидеть, как выглядят перемещения Петра во время случайного блуждания:

    ![Случайное блуждание Петра](ms-assets/ml/8-Reinforcement/1-QLearning/images/random_walk.gif)



## Функция награды

Чтобы сделать нашу политику более «умной», нам нужно понять, какие ходы «лучше» других. Для этого нужно определить нашу цель.

Цель можно выразить через **функцию награды**, которая возвращает численное значение для каждого состояния. Чем выше число, тем лучше награда. (кодовый блок 5)



In [ ]:
move_reward = -0.1
goal_reward = 10
end_reward = -10

def reward(m,pos=None):
    pos = pos or m.human
    if not m.is_valid(pos):
        return end_reward
    x = m.at(pos)
    if x==Board.Cell.water or x == Board.Cell.wolf:
        return end_reward
    if x==Board.Cell.apple:
        return goal_reward
    return move_reward

Интересная особенность функций награды заключается в том, что в большинстве случаев *нам даётся существенная награда только в конце игры*. Это означает, что наш алгоритм должен как-то запоминать «хорошие» шаги, которые приводят к положительной награде в конце, и повышать их значимость. Аналогично, все действия, ведущие к плохим результатам, должны поощряться меньше.



## Q-обучение

Алгоритм, который мы здесь рассмотрим, называется **Q-обучение**. В этом алгоритме политика определяется функцией (или структурой данных), называемой **Q-таблицей**. Она хранит «качество» каждого действия в каждом состоянии.

Её называют Q-таблицей, потому что удобно представить её в виде таблицы или многомерного массива. Поскольку наша доска имеет размеры `width` x `height`, мы можем представить Q-таблицу с помощью массива numpy с формой `width` x `height` x `len(actions)`: (кодовый блок 6)



In [ ]:
Q = np.ones((width,height,len(actions)),dtype=np.float64)*1.0/len(actions)

Обратите внимание, что все значения Q-таблицы инициализируются одинаковым значением, в нашем случае — 0.25. Это соответствует политике случайного блуждания, поскольку все действия в каждом состоянии равнозначны. Мы можем передать Q-таблицу в функцию `plot` для визуализации таблицы на доске: `m.plot(Q)`.

![Среда Петра](ms-assets/ml/env_init.04e8f26d2d60089e.webp)

В центре каждой ячейки показана «стрелка», указывающая предпочтительное направление движения. Поскольку все направления равны, отображается точка.

Теперь нам нужно запустить симуляцию, исследовать среду и обучить лучшее распределение значений Q-таблицы, что позволит нам гораздо быстрее находить путь к яблоку.



## Суть Q-обучения: уравнение Беллмана

Как только мы начнём движение, каждое действие будет иметь соответствующую награду, то есть мы теоретически можем выбрать следующее действие, основываясь на наивысшей немедленной награде. Однако во многих состояниях ход не приведёт к нашей цели — достижению яблока, и поэтому мы не можем сразу определить, какое направление лучше.

> Помните, что важен не немедленный результат, а итоговый результат, который мы получим в конце симуляции.

Чтобы учитывать эту отложенную награду, нам нужно использовать принципы **[динамического программирования](https://ru.wikipedia.org/wiki/Динамическое_программирование)**, позволяющие рекурсивно рассматривать нашу задачу.

Предположим, что мы сейчас в состоянии *s* и хотим перейти в следующее состояние *s'*. При этом мы получим немедленную награду *r(s,a)*, заданную функцией награды, плюс некоторую будущую награду. Если предположить, что наша Q-таблица корректно отражает «привлекательность» каждого действия, то в состоянии *s'* мы выберем действие *a*, соответствующее максимальному значению *Q(s',a')*. Таким образом, лучшая возможная будущая награда для состояния *s* определяется как `max`<sub>a'</sub>*Q(s',a')* (максимум берётся по всем возможным действиям *a'* в состоянии *s'*).

Это даёт **формулу Беллмана** для вычисления значения Q-таблицы в состоянии *s* при выборе действия *a*:

![](ms-assets/ml/bellman-equation.7c0c4c722e5a6b7c.webp)

Здесь γ — так называемый **коэффициент дисконтирования**, определяющий, в какой степени нужно отдавать предпочтение текущей награде перед будущей и наоборот.



## Алгоритм обучения

Исходя из приведённого уравнения, мы можем записать псевдокод нашего алгоритма обучения:

* Инициализировать Q-таблицу Q одинаковыми числами для всех состояний и действий
* Установить скорость обучения α ← 1
* Повторять симуляцию много раз
   1. Начать с случайной позиции
   1. Повторять
        1. Выбрать действие *a* в состоянии *s*
        2. Выполнить действие, перейдя в новое состояние *s'*
        3. Если достигнуто условие окончания игры или итоговая награда слишком мала — завершить симуляцию
        4. Вычислить награду *r* в новом состоянии
        5. Обновить Q-функцию согласно уравнению Беллмана: *Q(s,a)* ← *(1-α)Q(s,a)+α(r+γ max<sub>a'</sub>Q(s',a'))*
        6. *s* ← *s'*
        7. Обновить суммарную награду и уменьшить α.



## Использование vs исследование

В приведённом алгоритме не уточняется, как именно выбрать действие на шаге 2.1. Если выбирать действие случайно, то мы будем случайно **исследовать** среду, и, вероятно, часто погибать, а также исследовать области, куда мы обычно не зашли бы. Альтернативным подходом было бы **использовать** значения Q-таблицы, которые мы уже знаем, и выбирать лучшее действие (с наивысшим значением Q) в состоянии *s*. Однако это помешало бы исследовать другие состояния, и, вероятно, мы не нашли бы оптимальное решение.

Таким образом, лучший подход — найти баланс между исследованием и использованием. Это можно сделать, выбирая действие в состоянии *s* с вероятностями, пропорциональными значениям в Q-таблице. Вначале, когда все значения в Q-таблице одинаковы, это будет соответствовать случайному выбору, но по мере нашего обучения мы будем чаще следовать оптимальному маршруту, позволяя агенту иногда выбирать неизведанный путь.



## Реализация на Python

Теперь мы готовы реализовать алгоритм обучения. Перед этим нам нужна функция, которая преобразует произвольные числа из Q-таблицы в вектор вероятностей для соответствующих действий.

1. Создайте функцию `probs()`:

    ```python
    def probs(v,eps=1e-4):
        v = v-v.min()+eps
        v = v/v.sum()
        return v
    ```

    Мы добавляем немного `eps` к исходному вектору, чтобы избежать деления на 0 в начальном случае, когда все компоненты вектора одинаковы.

Запустите алгоритм обучения в течение 5000 экспериментов, также называемых **эпохами**: (кодовый блок 8)



In [ ]:
lpath = []
    for epoch in range(5000):
    
        # Выберите начальную точку
        m.random_start()
        
        # Начинайте движение
        n=0
        cum_reward = 0
        while True:
            x,y = m.human
            v = probs(Q[x,y])
            a = random.choices(list(actions),weights=v)[0]
            dpos = actions[a]
            m.move(dpos,check_correctness=False) # мы разрешаем игроку выходить за пределы доски, что завершает эпизод
            r = reward(m)
            cum_reward += r
            if r==end_reward or cum_reward < -1000:
                lpath.append(n)
                break
            alpha = max(0.05, np.exp(-epoch / 1000))  # скорость обучения убывает с номером эпохи
            gamma = 0.5
            ai = action_idx[a]
            Q[x,y,ai] = (1 - alpha) * Q[x,y,ai] + alpha * (r + gamma * Q[x+dpos[0], y+dpos[1]].max())
            n+=1

После выполнения этого алгоритма Q-таблица должна обновиться значениями, определяющими привлекательность разных действий на каждом шаге. Мы можем попытаться визуализировать Q-таблицу, нарисовав в каждой ячейке вектор, который указывает в желаемом направлении для движения. Для простоты вместо наконечника стрелки рисуем маленький кружок.

![](ms-assets/ml/learned.ed28bcd8484b5287.webp)



## Проверка политики

Поскольку Q-таблица показывает «привлекательность» каждого действия в каждом состоянии, довольно просто использовать её для определения эффективной навигации в нашем мире. В простейшем случае мы можем выбрать действие, соответствующее наивысшему значению Q-таблицы: (кодовый блок 9)



In [ ]:
def qpolicy_strict(m):
        x,y = m.human
        v = probs(Q[x,y])
        a = list(actions)[np.argmax(v)]
        return a

walk(m,qpolicy_strict)

> Если вы попробуете запустить код выше несколько раз, вы можете заметить, что иногда он "зависает", и вам приходится нажимать кнопку СТОП в блокноте, чтобы прервать выполнение. Это происходит потому, что могут быть ситуации, когда два состояния "смотрят" друг на друга с точки зрения оптимального Q-значения, в результате чего агент бесконечно перемещается между этими состояниями.



## Задача

> **Задание 1:** Измените функцию `walk`, чтобы ограничить максимальную длину пути определённым числом шагов (например, 100), и наблюдайте, как код выше время от времени возвращает это значение.

> **Задание 2:** Измените функцию `walk` так, чтобы она не возвращалась в места, где уже была ранее. Это предотвратит зацикливание функции `walk`, однако агент всё ещё может оказаться "запертым" в месте, из которого не сможет выбраться.



## Навигация

Лучшая навигационная политика — это та, которую мы использовали во время обучения, комбинирующая эксплуатацию и исследование. В этой политике мы выбираем каждое действие с определённой вероятностью, пропорциональной значениям в Q-таблице. Эта стратегия всё ещё может приводить к тому, что агент возвращается на уже исследованные позиции, но, как видно из кода ниже, она даёт очень короткий средний путь до нужного места (не забывайте, что `print_statistics` запускает симуляцию 100 раз): (блок кода 10)



In [ ]:
def qpolicy(m):
        x,y = m.human
        v = probs(Q[x,y])
        a = random.choices(list(actions),weights=v)[0]
        return a

print_statistics(qpolicy)

После запуска этого кода вы получите значительно меньшую среднюю длину пути, чем раньше, в диапазоне от 3 до 6.



## Исследование процесса обучения

Как мы уже упоминали, процесс обучения — это баланс между исследованием и использованием накопленных знаний о структуре пространства задачи. Мы видели, что результаты обучения (способность помогать агенту находить короткий путь к цели) улучшились, но также интересно наблюдать, как средняя длина пути ведёт себя во время процесса обучения:

![](ms-assets/ml/lpathlen1.0534784add58d4eb.webp)

Результаты обучения можно обобщить так:

- **Средняя длина пути увеличивается**. Здесь мы видим, что сначала средняя длина пути увеличивается. Это, вероятно, связано с тем, что, не зная ничего об окружении, агент с высокой вероятностью «застрять» в плохих состояниях, таких как вода или волк. По мере получения знаний мы можем исследовать окружение дольше, но всё ещё плохо знаем, где находятся яблоки.

- **Длина пути уменьшается по мере обучения**. Когда агент достаточно научился, ему становится легче достигать цели, и длина пути начинает сокращаться. Однако мы всё ещё не исключаем исследование, и агент часто отклоняется от лучшего пути, исследуя новые варианты, из-за чего путь становится длиннее оптимального.

- **Резкий рост длины пути**. Также на этом графике видно, что в какой-то момент длина пути резко увеличилась. Это свидетельствует о стохастической природе процесса и о том, что мы в какой-то момент можем "испортить" коэффициенты Q-таблицы, перезаписав их новыми значениями. Этого желательно избегать, уменьшая скорость обучения (например, к концу тренировки мы корректируем значения Q-таблицы малыми величинами).

В целом важно помнить, что успех и качество процесса обучения существенно зависят от параметров, таких как скорость обучения, затухание скорости обучения и коэффициент дисконтирования. Их часто называют **гиперпараметрами**, чтобы отличить от **параметров**, которые мы оптимизируем во время обучения (например, коэффициенты Q-таблицы). Процесс поиска наилучших значений гиперпараметров называется **оптимизацией гиперпараметров** и заслуживает отдельной темы.



## Задание

[Более реалистичный мир](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning/assignment.md)



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Discounted returns from rewards

●○○ · ~8 min

> In RL the return is G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + … — future rewards discounted by γ∈[0,1). Computing it backwards takes one O(n) pass: G_t = r_t + γ·G_{t+1}. Every value-learning method starts from this quantity.

Implement `discounted_returns(rewards, gamma)` → array `G` of the same length where `G[t]` is the discounted return from step `t` to the end of the episode.

<details><summary>Hint</summary>

Walk the list in reverse keeping a running total: acc = r[t] + gamma * acc, then store acc in G[t].

</details>


In [ ]:
import numpy as np

def discounted_returns(rewards, gamma):
    G = np.zeros(len(rewards))
    # your code
    return G

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    n = int(rng.integers(1, 20))   # length-1 episodes are legal\n    # gamma=0 (myopic — G_t = r_t) is a spec'd edge the lesson quiz tests\n    gamma = float(rng.choice([0.0, 0.0, rng.uniform(0.3, 0.99)]))\n    return {\"rewards\": rng.normal(0, 1, n), \"gamma\": gamma}\n\ndef discounted_returns(rewards, gamma):\n    r = np.asarray(rewards, dtype=float)\n    G = np.zeros_like(r)\n    acc = 0.0\n    for i in range(len(r) - 1, -1, -1):\n        acc = r[i] + gamma * acc\n        G[i] = acc\n    return G", "discounted_returns", 20)

### Task 2 — TD(0) value update

●○○ · ~8 min

> TD(0) bootstraps the value estimate: the target for V(s) is r + γ·V(s′) — or just r at a terminal transition — and the update steps toward it: V(s) ← V(s) + α·(target − V(s)). Unlike Monte-Carlo returns it learns from incomplete episodes, one transition at a time.

Implement `td_update(V, s, r, s2, alpha, gamma, done)` → new value vector (a copy; do not mutate the input). The TD target is `r` when `done` is True, otherwise `r + gamma * V[s2]`. Only entry `V[s]` changes.

<details><summary>Hint</summary>

Copy V, compute target = r if done else r + gamma * V[s2], then V[s] += alpha * (target - V[s]).

</details>


In [ ]:
import numpy as np

def td_update(V, s, r, s2, alpha, gamma, done):
    V = np.asarray(V, dtype=float).copy()
    # your code
    return V

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(3, 8))\n    return {\"V\": rng.normal(0, 1, S),\n            \"s\": int(rng.integers(S)), \"r\": float(rng.normal(0, 1)),\n            \"s2\": int(rng.integers(S)),\n            \"alpha\": float(rng.uniform(0.05, 0.6)),\n            \"gamma\": float(rng.uniform(0.7, 0.99)),\n            \"done\": bool(rng.random() < 0.25)}\n\ndef td_update(V, s, r, s2, alpha, gamma, done):\n    V = np.asarray(V, dtype=float).copy()\n    target = r if done else r + gamma * float(V[s2])\n    V[s] = V[s] + alpha * (target - V[s])\n    return V", "td_update", 20)

### Task 3 — Policy evaluation sweep

●●○ · ~12 min

> Policy evaluation computes the value of following a fixed policy π — how good each state is under it — via the Bellman expectation backup: V^π(s) = Σ_a π(a|s)·[R(s,a) + γ·Σ_{s'} T(s,a,s')·V(s')]. Here π(a|s) is the probability of taking action a in state s, R(s,a) the immediate reward, T(s,a,s') the probability of landing in s', and γ the discount. A synchronous sweep uses the old V everywhere. With matrices it is two contractions: Q = R + γ·(T @ V), then V' = Σ_a π·Q.

Implement `policy_eval_step(T, R, Pi, V, gamma)` → new value vector of shape (S,). `T` has shape (S, A, S) with T[s,a,s'] = P(s'|s,a), `R` and `Pi` have shape (S, A), `V` has shape (S,). One synchronous sweep: V'[s] = Σ_a Pi[s,a]·(R[s,a] + γ·Σ_s' T[s,a,s']·V[s']).

<details><summary>Hint</summary>

T @ V contracts the last axis and gives expected future value per (s,a) — shape (S,A). Then Q = R + gamma*(T@V) and return (Pi * Q).sum(axis=1).

</details>


In [ ]:
import numpy as np

def policy_eval_step(T, R, Pi, V, gamma):
    T = np.asarray(T, dtype=float)
    # your code
    return np.zeros(T.shape[0])

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(2, 6)); A = int(rng.integers(2, 5))\n    T = rng.random((S, A, S)); T = T / T.sum(axis=-1, keepdims=True)\n    Pi = rng.random((S, A)); Pi = Pi / Pi.sum(axis=-1, keepdims=True)\n    return {\"T\": T, \"R\": rng.normal(0, 1, (S, A)), \"Pi\": Pi,\n            \"V\": rng.normal(0, 1, S), \"gamma\": float(rng.uniform(0.7, 0.99))}\n\ndef policy_eval_step(T, R, Pi, V, gamma):\n    T = np.asarray(T, dtype=float); R = np.asarray(R, dtype=float)\n    Pi = np.asarray(Pi, dtype=float); V = np.asarray(V, dtype=float)\n    Q = R + gamma * (T @ V)\n    return (Pi * Q).sum(axis=1)", "policy_eval_step", 20)

### Task 4 — One Q-learning update

●●○ · ~10 min

> Q-learning bootstraps toward the TD target y = r + γ·max_{a'} Q(s',a') — or just r when the transition is terminal: Q(s,a) ← Q(s,a) + α·(y − Q(s,a)). One update touches exactly one entry of the table; all other entries stay unchanged.

Implement `q_update(Q, s, a, r, s2, alpha, gamma, done)` → updated Q-table (a new array, do not rely on in-place mutation). If `done` is True the target is `r`; otherwise `r + gamma * max(Q[s2])`. Update only entry `Q[s, a]`.

<details><summary>Hint</summary>

Copy Q, compute target = r if done else r + gamma * Q[s2].max(), then Q[s,a] += alpha * (target - Q[s,a]).

</details>


In [ ]:
import numpy as np

def q_update(Q, s, a, r, s2, alpha, gamma, done):
    Q = np.asarray(Q, dtype=float).copy()
    # your code
    return Q

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(3, 7)); A = int(rng.integers(2, 5))\n    return {\"Q\": rng.normal(0, 1, (S, A)),\n            \"s\": int(rng.integers(S)), \"a\": int(rng.integers(A)),\n            \"r\": float(rng.normal(0, 1)), \"s2\": int(rng.integers(S)),\n            \"alpha\": float(rng.uniform(0.05, 0.5)),\n            \"gamma\": float(rng.uniform(0.8, 0.99)),\n            \"done\": bool(rng.random() < 0.25)}\n\ndef q_update(Q, s, a, r, s2, alpha, gamma, done):\n    Q = np.asarray(Q, dtype=float).copy()\n    target = r if done else r + gamma * float(np.max(Q[s2]))\n    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])\n    return Q", "q_update", 20)

### Task 5 — One SARSA update

●●○ · ~10 min

> SARSA is **on-policy** TD control: the bootstrap target uses the action a′ the behaviour policy actually took, Q(s,a) ← Q(s,a) + α·(r + γ·Q(s′,a′) − Q(s,a)), with target r when the transition is terminal. Contrast with Q-learning's max over a′ — SARSA evaluates the policy being followed, exploration included.

Implement `sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, done)` → updated Q-table (a new array). If `done` the target is `r`; otherwise `r + gamma * Q[s2, a2]` — note it uses the given next action `a2`, not a max. Update only entry `Q[s, a]`.

<details><summary>Hint</summary>

Copy Q; target = r if done else r + gamma * Q[s2, a2]; then Q[s, a] += alpha * (target - Q[s, a]).

</details>


In [ ]:
import numpy as np

def sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, done):
    Q = np.asarray(Q, dtype=float).copy()
    # your code — build the target, then blend Q[s, a] toward it
    return Q

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(3, 7)); A = int(rng.integers(2, 5))\n    return {\"Q\": rng.normal(0, 1, (S, A)),\n            \"s\": int(rng.integers(S)), \"a\": int(rng.integers(A)),\n            \"r\": float(rng.normal(0, 1)),\n            \"s2\": int(rng.integers(S)), \"a2\": int(rng.integers(A)),\n            \"alpha\": float(rng.uniform(0.05, 0.5)),\n            \"gamma\": float(rng.uniform(0.8, 0.99)),\n            \"done\": bool(rng.random() < 0.25)}\n\ndef sarsa_update(Q, s, a, r, s2, a2, alpha, gamma, done):\n    Q = np.asarray(Q, dtype=float).copy()\n    target = r if done else r + gamma * float(Q[s2, a2])\n    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])\n    return Q", "sarsa_update", 20)

### Task 6 — Value iteration step

●●● · ~12 min

> Value iteration replaces the expectation over π with a max over actions: V_{k+1}(s) = max_a [R(s,a) + γ·Σ_{s'} T(s,a,s')·V_k(s)]. The greedy policy recovered afterwards is argmax_a of the same action-value terms — one sweep gives you both.

Implement `value_iteration_step(T, R, V, gamma)` → tuple `(V_new, policy)`. `T` has shape (S, A, S), `R` is (S, A), `V` is (S,). `V_new[s] = max_a (R[s,a] + γ·Σ_s' T[s,a,s']·V[s'])` and `policy[s]` is the argmax action index (lowest index on ties).

<details><summary>Hint</summary>

Same contraction as policy evaluation: Q = R + gamma * (T @ V); then return Q.max(axis=1), Q.argmax(axis=1).

</details>


In [ ]:
import numpy as np

def value_iteration_step(T, R, V, gamma):
    T = np.asarray(T, dtype=float)
    # your code
    return np.zeros(T.shape[0]), np.zeros(T.shape[0], dtype=int)

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(2, 6)); A = int(rng.integers(2, 5))\n    T = rng.random((S, A, S)); T = T / T.sum(axis=-1, keepdims=True)\n    return {\"T\": T, \"R\": rng.normal(0, 1, (S, A)),\n            \"V\": rng.normal(0, 1, S), \"gamma\": float(rng.uniform(0.7, 0.99))}\n\ndef value_iteration_step(T, R, V, gamma):\n    T = np.asarray(T, dtype=float); R = np.asarray(R, dtype=float)\n    V = np.asarray(V, dtype=float)\n    Q = R + gamma * (T @ V)\n    return Q.max(axis=1), Q.argmax(axis=1)", "value_iteration_step", 20)

### Task 7 — Epsilon-greedy action choice

●○○ · ~8 min

> ε-greedy is the simplest exploration strategy: with probability ε pick a uniformly random action, otherwise take the greedy argmax_a Q(s,a). For reproducibility the randomness must come from the generator passed in as an argument — draw the coin flip with rng.random() and the exploratory action with rng.integers(k).

Implement `epsilon_greedy(q_row, eps, rng)` → int action index. With probability `eps` return a uniform random action via `rng.integers(len(q_row))`; otherwise return `argmax` of the Q-row (ties break to the lowest index). Draw `rng.random()` first, exactly once, for the exploration check.

<details><summary>Hint</summary>

if rng.random() < eps: return int(rng.integers(len(q))); else return int(np.argmax(q)).

</details>


In [ ]:
import numpy as np

def epsilon_greedy(q_row, eps, rng):
    q = np.asarray(q_row, dtype=float)
    # your code
    return 0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    k = int(rng.integers(2, 7))\n    return {\"q_row\": rng.normal(0, 1, k),\n            \"eps\": float(rng.choice([0.0, 0.1, 0.3, 0.9, 1.0])),\n            \"rng\": rng}\n\ndef epsilon_greedy(q_row, eps, rng):\n    q = np.asarray(q_row, dtype=float)\n    if rng.random() < eps:\n        return int(rng.integers(len(q)))\n    return int(np.argmax(q))", "epsilon_greedy", 20)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/1-QLearning)
- [Учебник ML (ШАД): обучение с подкреплением](https://contest.yandex.ru/tracks/ml/environment-interaction/reinforcement-learning)